# Notebook 05 — Final Load Preparation & KPI Framework

**Objective:** Compute KPIs, create dashboard-ready aggregations, and export the final analytical dataset for Tableau visualisation.

**KPI Framework aligned to business problem:** Dealership pricing accuracy, market positioning, and inventory strategy.

In [28]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Libraries loaded. Output directory ready.")

Libraries loaded. Output directory ready.


## 1. Load Cleaned Dataset

In [29]:
CLEANED_PATH = '../data/processed/car_prices_cleaned.csv'

df = pd.read_csv(CLEANED_PATH)

print(f"Cleaned dataset loaded.")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Cleaned dataset loaded.
Shape: 548,486 rows x 29 columns

Columns (29):
['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'mmr', 'sellingprice', 'saledate', 'sale_year', 'sale_month', 'sale_month_name', 'sale_dow', 'sale_dow_name', 'vehicle_age', 'price_deviation', 'price_deviation_pct', 'price_realization_rate', 'sold_above_mmr', 'price_per_age_year', 'odometer_bucket', 'condition_tier']

First 3 rows:


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,sale_year,sale_month,sale_month_name,sale_dow,sale_dow_name,vehicle_age,price_deviation,price_deviation_pct,price_realization_rate,sold_above_mmr,price_per_age_year,odometer_bucket,condition_tier
0,2015,Kia,Sorento,Lx,Suv,Automatic,5xyktca69fg566472,ca,5.0000,"16,639.0000",White,Black,Kia Motors America Inc,"20,500.0000","21,500.0000",2014-12-16 12:30:00,"2,014.0000",12.0000,Dec,1.0000,Tue,0.0000,"1,000.0000",4.8800,104.8800,1,"21,500.0000",0–20k,Poor
1,2015,Kia,Sorento,Lx,Suv,Automatic,5xyktca69fg561319,ca,5.0000,"9,393.0000",White,Beige,Kia Motors America Inc,"20,800.0000","21,500.0000",2014-12-16 12:30:00,"2,014.0000",12.0000,Dec,1.0000,Tue,0.0000,700.0000,3.3700,103.3700,1,"21,500.0000",0–20k,Poor
2,2014,Bmw,3 Series,328I Sulev,Sedan,Automatic,wba3c1c51ek116351,ca,45.0000,"1,331.0000",Gray,Black,Financial Services Remarketing (Lease),"31,900.0000","30,000.0000",2015-01-15 04:30:00,"2,015.0000",1.0000,Jan,3.0000,Thu,1.0000,"-1,900.0000",-5.9600,94.0400,0,"30,000.0000",0–20k,Excellent


## 2. KPI Framework

| KPI | Formula | Business Use |
|-----|---------|-------------|
| Avg Price Realization Rate | sellingprice / mmr × 100 | Market competitiveness |
| Price Deviation % | (sellingprice - mmr) / mmr × 100 | Over/under-pricing signal |
| % Sold Above MMR | sold_above_mmr.mean() × 100 | Market demand strength |
| Avg Selling Price | sellingprice.mean() | Portfolio benchmark |
| Total Sales Volume | sellingprice.sum() | Revenue indicator |
| Avg Vehicle Age at Sale | vehicle_age.mean() | Inventory freshness |

In [30]:
# Compute all 6 portfolio-level KPIs
kpi_avg_price_realization = df['price_realization_rate'].mean()
kpi_avg_price_deviation_pct = df['price_deviation_pct'].mean()
kpi_pct_above_mmr = df['sold_above_mmr'].mean() * 100
kpi_avg_selling_price = df['sellingprice'].mean()
kpi_total_sales_volume = df['sellingprice'].sum()
kpi_avg_vehicle_age = df['vehicle_age'].mean()

print("=" * 60)
print("PORTFOLIO-LEVEL KPIs")
print("=" * 60)
print(f"  Avg Price Realization Rate : {kpi_avg_price_realization:.2f}%")
print(f"  Avg Price Deviation %      : {kpi_avg_price_deviation_pct:+.2f}%")
print(f"  % Sold Above MMR           : {kpi_pct_above_mmr:.1f}%")
print(f"  Avg Selling Price          : ${kpi_avg_selling_price:,.2f}")
print(f"  Total Sales Volume         : ${kpi_total_sales_volume:,.0f}")
print(f"  Avg Vehicle Age at Sale    : {kpi_avg_vehicle_age:.1f} years")

# Reference values
print("\n--- Reference Benchmarks ---")
print(f"  Total Transactions         : 558,675")
print(f"  Expected Total Volume      : ~$7.54B")
print(f"  Expected Avg Price         : ~$13,493")
print(f"  Expected % Above MMR       : 46.8%")

PORTFOLIO-LEVEL KPIs
  Avg Price Realization Rate : 99.30%
  Avg Price Deviation %      : -0.70%
  % Sold Above MMR           : 47.2%
  Avg Selling Price          : $13,577.60
  Total Sales Volume         : $7,447,121,112
  Avg Vehicle Age at Sale    : 4.8 years

--- Reference Benchmarks ---
  Total Transactions         : 558,675
  Expected Total Volume      : ~$7.54B
  Expected Avg Price         : ~$13,493
  Expected % Above MMR       : 46.8%


## 3. Make-Level Aggregation (for Tableau — Brand Performance View)

In [31]:
kpi_by_make = df.groupby('make').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean')
).reset_index()

kpi_by_make['pct_above_mmr'] = kpi_by_make['pct_above_mmr'] * 100
kpi_by_make = kpi_by_make.sort_values('transaction_count', ascending=False)

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_make.csv')
kpi_by_make.to_csv(output_path, index=False)

print(f"kpi_by_make.csv saved — {len(kpi_by_make)} makes")
print("\nTop 10 Makes by Transaction Count:")
print(kpi_by_make.head(10).to_string(index=False))

kpi_by_make.csv saved — 66 makes

Top 10 Makes by Transaction Count:
     make   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr
     Ford 14,481.8501 14,677.0167            -0.7614              93996        45.6222
Chevrolet 11,891.5543 12,059.5760            -0.7043              60581        47.4571
   Nissan 11,701.2349 11,828.2464            -0.8039              54017        47.2833
   Toyota 12,231.8474 12,340.9581            -0.4793              39964        46.5594
    Dodge 11,163.8442 11,372.8354            -1.8436              30953        46.2960
    Honda 10,905.4106 10,966.9985             0.1798              27351        49.8190
  Hyundai 11,001.9438 11,223.8302            -1.5538              21831        47.6845
      Bmw 20,865.5667 20,997.0423            -0.2958              20793        49.7235
      Kia 11,806.1754 11,944.7890            -1.1372              18082        47.0302
 Chrysler 11,068.1471 11,316.0070            -1.9415         

## 4. State-Level Aggregation (for Tableau — Regional Pricing View)

In [32]:
kpi_by_state = df.groupby('state').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean')
).reset_index()

kpi_by_state['pct_above_mmr'] = kpi_by_state['pct_above_mmr'] * 100
kpi_by_state = kpi_by_state.sort_values('transaction_count', ascending=False)

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_state.csv')
kpi_by_state.to_csv(output_path, index=False)

print(f"kpi_by_state.csv saved — {len(kpi_by_state)} states")
print("\nTop 10 States by Transaction Count:")
print(kpi_by_state.head(10).to_string(index=False))

kpi_by_state.csv saved — 38 states

Top 10 States by Transaction Count:
state   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr
   fl 13,836.0344 13,967.7621            -0.7462              81481        46.8170
   ca 14,219.3579 14,191.6338             1.8437              71662        55.1101
   pa 15,882.4838 16,048.8384            -1.1870              53158        45.0431
   tx 13,224.1744 13,445.0180            -2.3093              45195        47.3769
   ga 12,865.8917 12,894.2952             3.5875              34059        51.9980
   nj 13,577.8018 14,102.4531            -4.7973              27363        35.3324
   il 14,819.8724 14,985.4138            -1.8936              23188        44.9155
   nc  8,735.4142  8,632.0255             6.7694              21333        55.0134
   oh 14,263.0662 14,424.1813            -2.2987              21252        45.8874
   tn 17,007.9725 16,906.5548             0.7840              20691        54.1201


## 5. Condition Tier Aggregation (for Tableau — Condition Pricing View)

In [33]:
kpi_by_condition = df.groupby('condition_tier').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean'),
    avg_condition    = ('condition', 'mean')
).reset_index()

kpi_by_condition['pct_above_mmr'] = kpi_by_condition['pct_above_mmr'] * 100

# Order by condition tier
tier_order = ['Poor', 'Fair', 'Good', 'Excellent']
kpi_by_condition['condition_tier'] = pd.Categorical(
    kpi_by_condition['condition_tier'], categories=tier_order, ordered=True
)
kpi_by_condition = kpi_by_condition.sort_values('condition_tier')

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_condition.csv')
kpi_by_condition.to_csv(output_path, index=False)

print(f"kpi_by_condition.csv saved — {len(kpi_by_condition)} condition tiers")
print(kpi_by_condition.to_string(index=False))

kpi_by_condition.csv saved — 4 condition tiers
condition_tier   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr  avg_condition
          Poor 12,999.1981 13,630.7429            -7.7606              68223        39.0162         3.1997
          Fair  6,732.1263  7,604.0977            -8.7647              84873        25.6678        21.1631
          Good 11,096.0874 11,323.2684             1.0681             147003        42.6794        30.9866
     Excellent 17,544.1667 17,278.7564             2.9474             248387        59.4029        41.7649


## 6. Monthly Trend Aggregation (for Tableau — Temporal View)

In [34]:
kpi_monthly_trend = df.groupby(['sale_year', 'sale_month']).agg(
    transaction_count  = ('sellingprice', 'count'),
    total_volume       = ('sellingprice', 'sum'),
    avg_price          = ('sellingprice', 'mean'),
    avg_mmr            = ('mmr', 'mean'),
    avg_deviation_pct  = ('price_deviation_pct', 'mean'),
    pct_above_mmr      = ('sold_above_mmr', 'mean')
).reset_index()

kpi_monthly_trend['pct_above_mmr'] = kpi_monthly_trend['pct_above_mmr'] * 100
kpi_monthly_trend = kpi_monthly_trend.sort_values(['sale_year', 'sale_month'])

# Add month name for readability
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
kpi_monthly_trend['month_label'] = (
    kpi_monthly_trend['sale_month'].map(month_names) + ' ' +
    kpi_monthly_trend['sale_year'].astype(str)
)

output_path = os.path.join(PROCESSED_DIR, 'kpi_monthly_trend.csv')
kpi_monthly_trend.to_csv(output_path, index=False)

print(f"kpi_monthly_trend.csv saved — {len(kpi_monthly_trend)} month-year periods")
print(kpi_monthly_trend[['month_label','transaction_count','avg_price','pct_above_mmr']].to_string(index=False))

kpi_monthly_trend.csv saved — 10 month-year periods
month_label  transaction_count   avg_price  pct_above_mmr
 Jan 2014.0                206 15,555.9466        56.7961
 Feb 2014.0                  1 10,500.0000         0.0000
 Dec 2014.0              53423 11,208.6709        44.5314
 Jan 2015.0             138524 13,274.4305        46.0917
 Feb 2015.0             159114 13,629.6704        49.4966
 Mar 2015.0              45197 13,499.1104        51.4592
 Apr 2015.0               1405 10,319.6121        25.0534
 May 2015.0              51300 14,260.4402        41.5673
 Jun 2015.0              98037 14,892.3966        47.5667
 Jul 2015.0               1279 16,748.9163        48.0063


## 7. Body Type Aggregation (for Tableau — Vehicle Segment View)

In [35]:
kpi_by_body = df.groupby('body').agg(
    avg_price        = ('sellingprice', 'mean'),
    avg_mmr          = ('mmr', 'mean'),
    avg_deviation_pct= ('price_deviation_pct', 'mean'),
    transaction_count= ('sellingprice', 'count'),
    pct_above_mmr    = ('sold_above_mmr', 'mean'),
    avg_odometer     = ('odometer', 'mean'),
    avg_vehicle_age  = ('vehicle_age', 'mean')
).reset_index()

kpi_by_body['pct_above_mmr'] = kpi_by_body['pct_above_mmr'] * 100
kpi_by_body = kpi_by_body.sort_values('transaction_count', ascending=False)

output_path = os.path.join(PROCESSED_DIR, 'kpi_by_body.csv')
kpi_by_body.to_csv(output_path, index=False)

print(f"kpi_by_body.csv saved — {len(kpi_by_body)} body types")
print("\nTop 15 Body Types by Transaction Count:")
print(kpi_by_body.head(15).to_string(index=False))

kpi_by_body.csv saved — 46 body types

Top 15 Body Types by Transaction Count:
        body   avg_price     avg_mmr  avg_deviation_pct  transaction_count  pct_above_mmr  avg_odometer  avg_vehicle_age
       Sedan 11,607.7875 11,782.2225            -1.1245             241649        46.3238   63,211.6643           4.5565
         Suv 15,936.7037 16,079.4652            -0.5709             144474        47.9546   72,429.8398           4.9932
   Hatchback 10,032.2543 10,188.6668            -1.2408              26248        47.0169   52,067.2958           3.8850
     Minivan 11,484.1909 11,609.2777             0.0370              25750        45.6194   75,907.5127           4.6447
       Coupe 15,137.9024 15,240.7485            -0.3816              17785        49.6654   67,772.8684           5.8142
       Wagon 10,027.7747 10,152.4890            -1.4395              16422        47.3694   75,507.1241           5.2752
    Crew Cab 21,599.2292 21,720.4932             0.3870              16403

## 8. Dashboard Data Summary

| File | Rows | Purpose |
|------|------|--------|
| car_prices_cleaned.csv | 558,675 | Full transaction-level data |
| kpi_by_make.csv | ~66 | Brand performance dashboard |
| kpi_by_state.csv | ~50 | Regional pricing map |
| kpi_by_condition.csv | 4 | Condition tier analysis |
| kpi_monthly_trend.csv | ~24 | Temporal trend chart |
| kpi_by_body.csv | ~46 | Vehicle segment comparison |

In [36]:
# Verify all output files exist
output_files = [
    'car_prices_cleaned.csv',
    'kpi_by_make.csv',
    'kpi_by_state.csv',
    'kpi_by_condition.csv',
    'kpi_monthly_trend.csv',
    'kpi_by_body.csv'
]

print("=" * 60)
print("OUTPUT FILE VERIFICATION")
print("=" * 60)
for fname in output_files:
    fpath = os.path.join(PROCESSED_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  [OK] {fname:<35} {size_kb:>8.1f} KB")
    else:
        print(f"  [--] {fname:<35} (not yet generated)")

print()
print("✅ All KPI aggregation files exported to data/processed/. Load into Tableau Public for dashboard build.")

OUTPUT FILE VERIFICATION
  [OK] car_prices_cleaned.csv              119048.8 KB
  [OK] kpi_by_make.csv                          4.9 KB
  [OK] kpi_by_state.csv                         3.1 KB
  [OK] kpi_by_condition.csv                     0.5 KB
  [OK] kpi_monthly_trend.csv                    1.2 KB
  [OK] kpi_by_body.csv                          5.3 KB

✅ All KPI aggregation files exported to data/processed/. Load into Tableau Public for dashboard build.
